<a href="https://colab.research.google.com/github/mc-castro/clinicaltrials-ia-thesis/blob/mc-castro%2Fdevelop/notebooks/feature_engineering_mimic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [45]:
import pandas as pd
from google.colab import drive
import os

In [46]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Data

In [47]:
DATA_RAW_PATH = '/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/raw/'

# Full diagnosis codes (ICD codes and titles)
df_diag_all = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_diagnoses_all_true.csv'))

# Core admission info (textual data, identifiers)
df_admissions = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_diagnoses.csv'),
                            usecols=['hadm_id', 'subject_id', 'chief_complaint', 'physical_exam', 'HPI'])

# Procedures performed
df_procedures = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_procedures.csv'))

# Laboratory events table (first occurrence only)
df_lab = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_labevents_first_lab.csv'))

# Pacientes info
df_patients = pd.read_csv(os.path.join(DATA_RAW_PATH, 'patients.csv'), usecols=['subject_id', 'gender', 'anchor_age', 'dod'])

## Transform

### Main Diagnosis Processing

In [48]:
# Filter the main diagnosis (seq_num = 1) and extract the code and name
df_diag_principal = df_diag_all[df_diag_all['seq_num'] == 1][['hadm_id', 'icd_code', 'long_title']]
df_diag_principal.rename(columns={'icd_code': 'icd_diag_principal', 'long_title': 'name_diag_principal'}, inplace=True)

### Processing of Procedures

In [49]:
# Groups procedures into a single line by admission (hadm_id)
df_procs_grouped = df_procedures.groupby('hadm_id')['long_title'].apply(lambda x: ', '.join(x.astype(str).unique())).reset_index()
df_procs_grouped.rename(columns={'long_title': 'list_procedures'}, inplace=True)

### Process Patientes

In [50]:
df_patients = df_patients.rename(columns={'anchor_age': 'age'})
df_patients['age'] = df_patients['age'].astype(int)
df_patients['is_dead'] = df_patients['dod'].notna()
df_patients = df_patients[df_patients['is_dead'] == False]
df_patients.drop(columns=['dod', 'is_dead'], inplace=True)

### Union Tables

In [51]:
# 1. Join Admission with the Main Diagnosis
df_base = pd.merge(df_admissions, df_diag_principal, on='hadm_id', how='left')

# 2. Join the result with the Procedures
df_final_base = pd.merge(df_base, df_procs_grouped, on='hadm_id', how='left')

In [52]:
print("\n--- Base DataFrame (Demographics + Diagnosis + Procedures) Created ---")
print(f"Number of unique hospital admissions: {df_final_base['hadm_id'].nunique()}")


--- Base DataFrame (Demographics + Diagnosis + Procedures) Created ---
Number of unique hospital admissions: 4761


### Attaching Lab Events (Pivoting)

In [53]:
df_lab.value_counts('flag')

,count
flag,
abnormal,61790


In [54]:
# Apply the text parsing for CRP/critical values (from previous step)
def handle_c_reactive_protein(row):
    if pd.notna(row['valuenum']): return row['valuenum']
    comment = str(row['comments']).upper()
    if 'GREATER THAN 300' or '>300.' in comment: return 301.0
    return row['valuenum']

In [55]:
# Select relevant columns and filter to keep only rows where the fluid is 'Blood' (systemic eligibility criteria are almost always based on Blood measurements)
df_lab_clean = df_lab.loc[df_lab['fluid'] == 'Blood', ['hadm_id', 'label', 'valuenum', 'flag', 'comments']].copy()
df_lab_clean['hadm_id'] = df_lab_clean['hadm_id'].astype(int)

# Use and clean up the original 'flag' for robustness
df_lab_clean['flag'] = df_lab_clean['flag'].fillna('normal')

CRP_LABEL = 'C-Reactive Protein'
df_lab_clean['valuenum'] = df_lab.apply(lambda row:
                                          handle_c_reactive_protein(row)
                                          if row['label'] == CRP_LABEL else row['valuenum'],
                                          axis=1)

# PCR > 300 should be always abnormal
df_lab_clean.loc[(df_lab_clean['label'] == CRP_LABEL) & (df_lab_clean['valuenum'] == 301.0), 'flag'] = 'abnormal'

# Pivot 1: make each unique 'label' (lab test) a new column
# The 'valuenum' becomes the value in the new column
df_lab_pivot_value = df_lab_clean.pivot(index='hadm_id', columns='label', values='valuenum')
df_lab_pivot_value.reset_index(inplace=True)
df_lab_pivot_value.head()

# PIVOT 2: Status Flag (e.g., Creatinine_FLAG: ABNORMAL)
df_lab_pivot_flag = df_lab_clean.pivot_table(
    index='hadm_id',
    columns='label',
    values='flag',
    aggfunc='first' # Get the first recorded status text
)
df_lab_pivot_flag.reset_index(inplace=True)

# Rename columns from PIVOT 2 to avoid collision
df_lab_pivot_flag.columns = [col + '_FLAG' if col != 'hadm_id' else col for col in df_lab_pivot_flag.columns]



In [56]:
df_lab_clean.value_counts('flag')

,count
flag,
normal,128761
abnormal,57478


### Final Merge

In [57]:
# Merge Base DF (Demographics) with both Pivot Value and Pivot Flag DFs
df_final_complete = pd.merge(df_final_base, df_lab_pivot_value, on='hadm_id', how='left')
df_final_complete = pd.merge(df_final_complete, df_lab_pivot_flag, on='hadm_id', how='left')
df_final_complete = pd.merge(df_final_complete, df_patients, on='subject_id', how='inner')

In [58]:
print("\n--- df_final_complete Created (Dual Pivot) ---")
print(f"Total number of columns: {len(df_final_complete.columns)}")
print(f"Total number of rows: {len(df_final_complete)}")


--- df_final_complete Created (Dual Pivot) ---
Total number of columns: 682
Total number of rows: 2813


## Load

In [59]:
DATA_PROCESSED_PATH = '/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/'
OUTPUT_FILENAME = 'df_diag_final_elegibility.parquet'
OUTPUT_PATH = os.path.join(DATA_PROCESSED_PATH, OUTPUT_FILENAME)

try:
    df_final_complete.to_parquet(OUTPUT_PATH, index=False)
    print(f"\n--- SUCESSO: DataFrame final salvo em: {OUTPUT_PATH} ---")
except Exception as e:
    print(f"\nERRO ao salvar o arquivo: {e}")


--- SUCESSO: DataFrame final salvo em: /content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/df_diag_final_elegibility.parquet ---
